## Tujuan

Tahap ini bertujuan untuk membersihkan dataset dari data yang tidak valid serta melakukan transformasi dan pembuatan fitur baru. Proses ini dilakukan agar data menjadi lebih konsisten dan informatif untuk digunakan pada tahap analisis dan pemodelan.

In [30]:
import pandas as pd
import numpy as np

data = "../data/teen_mental_health.csv"

df = pd.read_csv(data)

### 1. Data Cleaning & Inspeksi Awal
Langkah pertama adalah melihat sekilas isi data dan memastikan tidak ada data yang kosong (*missing values*). Kita juga akan mengecek tipe data masing-masing kolom.

In [31]:
df.isnull().sum()

age                         0
gender                      0
daily_social_media_hours    0
platform_usage              0
sleep_hours                 0
screen_time_before_sleep    0
academic_performance        0
physical_activity           0
social_interaction_level    0
stress_level                0
anxiety_level               0
addiction_level             0
depression_label            0
mental_health_risk_score    0
sleep_quality               0
digital_wellbeing_flag      0
dtype: int64

In [32]:
df.head()

,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label,mental_health_risk_score,sleep_quality,digital_wellbeing_flag
0,14,male,7.9,Facebook,7.4,2.9,3.01,1.5,low,2,2,1,0,5,Fair,At Risk
1,19,female,1.9,TikTok,8.0,2.9,3.22,0.8,high,8,1,10,0,19,Good,Moderate
2,17,female,1.3,Instagram,7.6,0.5,3.92,0.0,high,2,4,2,0,8,Fair,Healthy
3,15,male,7.4,YouTube,6.9,1.6,3.48,0.8,medium,1,7,9,0,17,Fair,Moderate
4,15,female,4.7,All Platforms,4.9,3.0,2.37,1.4,medium,3,5,2,0,10,Poor,Moderate


Kolom *platform_usage* dihapus karena tidak bersifat deterministik terhadap kondisi kesehatan mental. Faktor yang lebih berpengaruh adalah aspek perilaku seperti durasi penggunaan, kualitas tidur, serta tingkat stres dan kecanduan, sehingga fitur berbasis perilaku dinilai lebih relevan untuk pemodelan.

In [33]:
df = df.drop(columns=['platform_usage'])

Beberapa kolom berupa teks perlu diubah ke tipe data `category` untuk menghemat memori. Selain itu, kita perlu memastikan isi teksnya seragam, misalnya mengubah semua huruf pada kolom `social_interaction_level` menjadi huruf kecil (*lowercase*) untuk menghindari duplikasi kategori seperti 'Low' dan 'low'.

In [34]:
# mengubah tipe data ke kategori
cat_cols = ['gender', 'sleep_quality', 'digital_wellbeing_flag']
for col in cat_cols:
    df[col] = df[col].astype('category')

# menyeragamkan huruf menjadi huruf kecil pada kolom teks
df['social_interaction_level'] = df['social_interaction_level'].str.lower()

# cek hasil nilai unik pada kolom teks untuk memastikan sudah seragam
for col in df.select_dtypes(include=['object', 'category']):
    print(f"Nilai unik {col}: {df[col].unique().tolist()}")

Nilai unik gender: ['male', 'female']
Nilai unik social_interaction_level: ['low', 'high', 'medium']
Nilai unik sleep_quality: ['Fair', 'Good', 'Poor']
Nilai unik digital_wellbeing_flag: ['At Risk', 'Moderate', 'Healthy']


### Feature Engineering

Pada ini kita menambahkan fitur baru dari kombinasi data yang sudah ada agar informasi yang dihasilkan lebih kaya dan mudah dianalisis.

1. **`total_screen_exposure`**  
   Menggambarkan total waktu yang dihabiskan di depan layar dalam sehari, yaitu gabungan dari durasi penggunaan media sosial dan *screen time* sebelum tidur.

2. **`sleep_efficiency`**  
   Menunjukkan perbandingan antara jam tidur dengan *screen time* sebelum tidur. Semakin tinggi nilainya, diasumsikan kualitas tidur semakin baik. Penambahan angka 1 pada pembagi dilakukan untuk menghindari pembagian dengan nol.

3. **`high_social_media_usage`**  
   Fitur indikator (1 atau 0) yang menandai apakah penggunaan media sosial melebihi 5 jam per hari.

4. **`active_lifestyle`**  
   Fitur indikator (1 atau 0) untuk menunjukkan apakah individu memiliki aktivitas fisik minimal 1 jam per hari.

5. **`risk_category`**  
   Mengelompokkan nilai `mental_health_risk_score` ke dalam tiga kategori, yaitu Low, Medium, dan High, agar lebih mudah dianalisis dan diinterpretasikan.

In [35]:
# 1. Total paparan layar
df['total_screen_exposure'] = df['daily_social_media_hours'] + df['screen_time_before_sleep']

# 2. Efisiensi tidur
df['sleep_efficiency'] = df['sleep_hours'] / (df['screen_time_before_sleep'] + 1)

# 3. Flagging kebiasaan main sosmed tinggi (> 5 jam)
df['high_social_media_usage'] = (df['daily_social_media_hours'] > 5).astype(int)

# 4. Flagging gaya hidup aktif (Aktivitas fisik >= 1 jam)
df['active_lifestyle'] = (df['physical_activity'] >= 1).astype(int)

# 5. Kategorisasi Tingkat Risiko
def risk_category(x):
    if x < 10:
        return 'low'
    elif x < 20:
        return 'medium'
    else:
        return 'high'

df['risk_category'] = df['mental_health_risk_score'].apply(risk_category)

print("[INFO] Fitur baru berhasil ditambahkan!")
df[['total_screen_exposure', 'sleep_efficiency', 'high_social_media_usage', 'active_lifestyle', 'risk_category']].head()

[INFO] Fitur baru berhasil ditambahkan!


,total_screen_exposure,sleep_efficiency,high_social_media_usage,active_lifestyle,risk_category
0,10.8,1.897436,1,1,low
1,4.8,2.051282,0,0,medium
2,1.8,5.066667,0,0,low
3,9.0,2.653846,1,0,medium
4,7.7,1.225000,0,1,medium


Dikarenakan model Machine Learning tidak mengerti teks. Oleh karena itu, kita harus mengubah teks menjadi angka:
* **Label Encoding**: Digunakan untuk data yang memiliki tingkatan/hierarki (Ordinal). Contoh: `sleep_quality` (Poor < Fair < Good).
* **One-Hot Encoding**: Digunakan untuk data yang setara/tidak bertingkat (Nominal). Contoh: `gender`. Kita memecahnya menjadi beberapa kolom biner (0/1).

In [36]:
# A. Label Encoding (Ordinal)
sleep_map = {'Poor': 0, 'Fair': 1, 'Good': 2}
df['sleep_quality_encoded'] = df['sleep_quality'].map(sleep_map)

# B. One-Hot Encoding (Nominal)
# drop_first=True digunakan untuk menghindari Dummy Variable Trap (Multikolinearitas)
df = pd.get_dummies(df, columns=['gender'], drop_first=True) 

print("[INFO] Encoding berhasil. Struktur kolom saat ini:")
print(df.columns.tolist())

[INFO] Encoding berhasil. Struktur kolom saat ini:
['age', 'daily_social_media_hours', 'sleep_hours', 'screen_time_before_sleep', 'academic_performance', 'physical_activity', 'social_interaction_level', 'stress_level', 'anxiety_level', 'addiction_level', 'depression_label', 'mental_health_risk_score', 'sleep_quality', 'digital_wellbeing_flag', 'total_screen_exposure', 'sleep_efficiency', 'high_social_media_usage', 'active_lifestyle', 'risk_category', 'sleep_quality_encoded', 'gender_male']


Data yang sudah rapi dan diperkaya dengan fitur baru ini akan disimpan ke dalam file baru. File ini yang nantinya akan digunakan di notebook selanjutnya (EDA dan Pemodelan).

In [37]:
df.to_csv('../data/clean_data.csv', index=False)

## Kesimpulan

Pada tahap ini, dataset kesehatan mental remaja telah berhasil dibersihkan dan distandardisasi tipe datanya. Kita juga telah berhasil mengekstraksi **5 fitur baru** (*Feature Engineering*) seperti `total_screen_exposure`, `sleep_efficiency`, hingga `risk_category` yang diharapkan mampu memberikan konteks lebih kuat bagi model prediktif nantinya. Variabel teks juga sudah dikonversi menjadi format numerik menggunakan metode *Label Encoding* dan *One-Hot Encoding*. 

Dataset sudah siap untuk dianalisis lebih dalam pada tahap **Exploratory Data Analysis (EDA)**.